In [49]:
import os
from langgraph.graph import StateGraph, MessagesState, START, END
from typing import TypedDict, Annotated, List
from langchain_groq import ChatGroq
from langchain_classic import PromptTemplate

In [50]:
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [ ]:
import pandas as pd

df = pd.read_csv("house_price.csv")

unit_multiplier = {"Cr": 100, "L": 1}
df["price_lakh"] = (
    (df["price"] * df["price_unit"].map(unit_multiplier)).round().astype("int64")
)

df = df.drop(columns=["Unnamed: 0", "price", "price_unit"])

print(df.head())

   bhk       type  area   region              status  price_lakh
0    3  Apartment  1076  Andheri  Under Construction         319
1    3  Apartment  1076  Andheri  Under Construction         319
2    3  Apartment  1076  Andheri  Under Construction         319
3    2  Apartment   718  Andheri  Under Construction         212
4    3  Apartment  1076  Andheri  Under Construction         319


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5042 entries, 0 to 5041
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   bhk         5042 non-null   int64
 1   type        5042 non-null   str  
 2   area        5042 non-null   int64
 3   region      5042 non-null   str  
 4   status      5042 non-null   str  
 5   price_lakh  5042 non-null   int64
dtypes: int64(3), str(3)
memory usage: 236.5 KB


In [11]:
df.head()

,bhk,type,area,region,status,price_lakh
0,3,Apartment,1076,Andheri,Under Construction,319
1,3,Apartment,1076,Andheri,Under Construction,319
2,3,Apartment,1076,Andheri,Under Construction,319
3,2,Apartment,718,Andheri,Under Construction,212
4,3,Apartment,1076,Andheri,Under Construction,319


In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5042 entries, 0 to 5041
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   bhk         5042 non-null   int64
 1   type        5042 non-null   str  
 2   area        5042 non-null   int64
 3   region      5042 non-null   str  
 4   status      5042 non-null   str  
 5   price_lakh  5042 non-null   int64
dtypes: int64(3), str(3)
memory usage: 236.5 KB


In [13]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///house_prices.db")

In [14]:
df.to_sql("house_prices", engine, if_exists="replace", index=False)

5042

In [15]:
from sqlalchemy import text

with engine.connect() as conn:
    result = conn.execute(text("SELECT * FROM house_prices LIMIT 5"))
    for row in result:
        print(row)

(3, 'Apartment', 1076, 'Andheri', 'Under Construction', 319)
(3, 'Apartment', 1076, 'Andheri', 'Under Construction', 319)
(3, 'Apartment', 1076, 'Andheri', 'Under Construction', 319)
(2, 'Apartment', 718, 'Andheri', 'Under Construction', 212)
(3, 'Apartment', 1076, 'Andheri', 'Under Construction', 319)


In [20]:
from sqlalchemy import inspect

inspector = inspect(engine)

tables = inspector.get_table_names()

schema = {}

for table in tables:
    columns = inspector.get_columns(table)
    schema[table] = [col["name"] for col in columns]

print(schema)

{'house_prices': ['bhk', 'type', 'area', 'region', 'status', 'price_lakh']}


In [ ]:
def run_query(sql):
    with engine.connect() as conn:
        result = conn.execute(text(sql))
        return result.fetchall()


print(
    run_query(
        """SELECT region, AVG(price_lakh) 
                FROM house_prices 
                GROUP BY region;"""
    )
)

[('Andheri', 281.25324675324674), ('Bandra', 613.859756097561), ('Dadar', 533.4423076923077), ('Ghatkopar', 177.1214351425943), ('Lower Parel', 813.9481707317074), ('Mumbai', 30.0)]


In [ ]:
class SQLState(TypedDict):
    query: str

In [51]:
template = PromptTemplate.from_template(
    """
    You are an SQL expert.

    Schema:
    {schema}

    Convert question to SQL and dont add any extra contents also do with proper syntax,dont add any extra charcters:
    {question}
    """
)

In [52]:
chain = template | llm

In [59]:
op = chain.invoke({"schema": schema, "question": "Find out the avg price?"})

In [60]:
print(op)

content='SELECT AVG(price_lakh) FROM house_prices' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 105, 'total_tokens': 115, 'completion_time': 0.022709256, 'completion_tokens_details': None, 'prompt_time': 0.014658525, 'prompt_tokens_details': None, 'queue_time': 0.332975582, 'total_time': 0.037367781}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d6899-f290-7fb1-afd7-0e5a786c4b14-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 105, 'output_tokens': 10, 'total_tokens': 115}


In [61]:
def run_query(sql):
    with engine.connect() as conn:
        result = conn.execute(text(sql))
        return result.fetchall()


print(run_query(op.content))

[(336.2669575565252,)]
